In [3]:
import yfinance as yf 
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
def check_type(data):
    first_prompt = data['prompt'].loc[data['prompt'] != 0].iloc[0]
    return 1 if first_prompt == 1 else 0
def returns(data):
    if(check_type(data)==1):
        returns_p=[]
        for i in range(0,len(data)):
            if(data['prompt'].iloc[i]==1):
                for j in range(i+1,len(data)):
                    if(data['prompt'].iloc[j]==-1):
                        returns_p.append((data['Close'].iloc[j]-data['Close'].iloc[i])*100/data['Close'].iloc[i])
                        i=j
                        break
        returns_p=np.array(returns_p)
        return returns_p
    elif(check_type(data)==0):
        returns_p=[]
        for i in range(0,len(data)):
            if(data['prompt'].iloc[i]==-1):
                for j in range(i+1,len(data)):
                    if(data['prompt'].iloc[j]==1):
                        returns_p.append((data['Close'].iloc[i]-data['Close'].iloc[j])*100/data['Close'].iloc[j])
                        i=j
                        break
        returns_p=np.array(returns_p)
        return returns_p
def no_trades(data):
    return(len(returns(data)))
    
def MaxDrawDown(data):
    
#     max_peak_till_now = data["Close"].cummax()
#     drawdown = (data["Close"] - max_peak_till_now)/max_peak_till_now
#     return drawdown.min()*100
    final=np.zeros(no_trades(data))
    flag=0
    if(check_type(data)==1):
       
        for i in range(0,len(data)):
            if(data['prompt'].iloc[i]==1):
                 for j in range(i+1,len(data)):
                         if(data['prompt'].iloc[j]==-1):
                                max_peak_till_now = data["Close"].iloc[i:j+1].cummax()
                                drawdown = (data["Close"].iloc[i:j+1] - max_peak_till_now)/max_peak_till_now
                                final[flag]=drawdown.min()*100
                                flag=flag+1
                                break
                                
    elif(check_type(data)==0):
        for i in range(0,len(data)):
            if(data['prompt'].iloc[i]==-1):
                 for j in range(i+1,len(data)):
                         if(data['prompt'].iloc[j]==1):
                                min_low_till_now = data["Close"].iloc[i:j+1].cummin()
                                drawdown = (min_low_till_now-data["Close"].iloc[i:j+1])/min_low_till_now
                                final[flag]=drawdown.min()*100
                                flag=flag+1
                                break
    return final.min()
                        
                
            
            

            
def SharpeRatio(data):
    
    rfr = 0
    return(252 * (returns(data).mean() - rfr )/(np.sqrt(252) * returns(data).std()))

def portfolio_value(data):
    capital=10000000
 
    if(check_type(data)==0):
        for i in range(0,len(data)):
            if(data['prompt'].iloc[i]==-1):
                for j in range(0,len(data)):
                    if(data['prompt'].iloc[j]==1):
                        capital=(capital//data['Close'].iloc[j])*(data['Close'].iloc[i]) +capital%(data['Close'].iloc[j])
                        break
                        
    elif(check_type(data)==1):
           for i in range(0,len(data)):
            if(data['prompt'].iloc[i]==1):
                for j in range(0,len(data)):
                    if(data['prompt'].iloc[j]==-1):
                        capital=(capital//data['Close'].iloc[i])*(data['Close'].iloc[j]) +capital%(data['Close'].iloc[i])
                        break
        
    return capital

data=yf.download('^NSEI',start='2018-01-01',end='2024-01-01')
data['sma']=data['Close'].rolling(window=20).mean()
data['upper']=data['sma']+2*(data['Close'].rolling(window=20).std())
data['lower']=data['sma']-2*(data['Close'].rolling(window=20).std())
data['prompt']=np.zeros(len(data))
for i in range(0,len(data)):
    if(data['Close'].iloc[i]>=data['upper'].iloc[i] and data['Open'].iloc[i]<=data['upper'].iloc[i]):
        data['prompt'].iloc[i]=-1
    elif(data['Close'].iloc[i]<=data['lower'].iloc[i] and data['Open'].iloc[i]>=data['lower'].iloc[i]):
        data['prompt'].iloc[i]=1
        
# b=data['prompt'].where(data['prompt']!=0)
# print(data)

for i in range(0,len(data)):
    if(data['prompt'].iloc[i]==1):
        for j in range(i+1,len(data)):
            if(data['prompt'].iloc[j]==-1):
                i=j
                break
            elif(data['prompt'].iloc[j]==1):
                data['prompt'].iloc[j]=0
for i in range(0,len(data)):
    if(data['prompt'].iloc[i]==-1):
        for j in range(i+1,len(data)):
            if(data['prompt'].iloc[j]==-1):
                data['prompt'].iloc[j]=0
            elif(data['prompt'].iloc[j]==1):
                i=j
                break


                
flag=data['prompt'].sum()

        
if(flag==1):
    data['prompt'].iloc[-1]=-1
elif(flag==-1):
    data['prompt'].iloc[-1]=1
print(flag)    
print(data.loc[:, ~data.columns.isin([ 'High','Low','Volume','Adj Close','sma'])].to_string())
print('THE NUMBER OF TRADES TAKEN IS:',no_trades(data))
print('RETURNS ON EVERY TRADE IS:',returns(data))
print('THE PORTFOLIO VALUE IS:',portfolio_value(data))
print('THE MAX DRAWDOWN IS:',MaxDrawDown(data),'%')
print('THE SHARPE RATIO IS:',SharpeRatio(data))

[*********************100%%**********************]  1 of 1 completed


1.0
                    Open         Close         upper         lower  prompt
Date                                                                      
2018-01-02  10477.549805  10442.200195           NaN           NaN     0.0
2018-01-03  10482.650391  10443.200195           NaN           NaN     0.0
2018-01-04  10469.400391  10504.799805           NaN           NaN     0.0
2018-01-05  10534.250000  10558.849609           NaN           NaN     0.0
2018-01-08  10591.700195  10623.599609           NaN           NaN     0.0
2018-01-09  10645.099609  10637.000000           NaN           NaN     0.0
2018-01-10  10652.049805  10632.200195           NaN           NaN     0.0
2018-01-11  10637.049805  10651.200195           NaN           NaN     0.0
2018-01-12  10682.549805  10681.250000           NaN           NaN     0.0
2018-01-15  10718.500000  10741.549805           NaN           NaN     0.0
2018-01-16  10761.500000  10700.450195           NaN           NaN     0.0
2018-01-17  10702.450

THE SHARPE RATIO IS: 8.512525416595059
